# <div>
# <img src="https://i.imgur.com/rqmo4C0.png" width=200/>
# </div>

# Lesson 11 - Retrieval Augmented Generation (RAG)

By the end of this notebook, you should be able to answer the following questions:

- What is <b>fine-tuning</b>? What is <b>RAG</b>? What's the difference between the two?</li>
- When should you pick fine-tuning vs RAG for a specific use-case?</li>
- How do <b>vector databases</b> enable semantic search in RAG systems?</li>

## 0. Prerequisites & Setup

- Make sure you have created a clean Python environment & you are currently inside it.
- Make sure your Python version is 3.8 or above.

In [ ]:
# OpenAI & tokenizer
!pip install openai tiktoken

# Agent packages & tools
!pip install langchain langchain-experimental langchain-openai

# RAG packages & tools
!pip install lancedb sentence-transformers tantivy beautifulsoup4 ragas

In [ ]:
import numpy as np
import random
import os

from getpass import getpass

from openai import OpenAI

from langchain.chat_models import init_chat_model

# RAG-related imports
import lancedb

from langchain_community.vectorstores import LanceDB
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from numpy import dot, linalg

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

OPENAI_MODEL = "gpt-4o"
OPENAI_MODEL_SMALL = "gpt-4o-mini"
OPENAI_MODEL_LEGACY = "gpt-3.5-turbo"
OPENAI_CLIENT = OpenAI() # Reads the API key from the environment variable

## 1. LLM Fine-tuning & RAG

### 1.1. Overview

<div style="width: 80%; text-align: justify">

LLM foundation models have a vast amount of <b>general knowledge</b>. However, this knowledge is restricted to the data that the model was trained on - as such, each model has a <b>knowledge cutoff date</b>, a point after which the model doesn't know anything. For example, a model trained in 2023 does not know anything about events from 2024. As a result, it might admit its shortcomings or, in the worst case scenario, hallucinate incorrect data in a confident manner.

Another limitation regarding LLMs is in the case of <b>domain-specific</b> knowledge, such as data about smaller cities, pay-walled articles, or business-specific data.

While prompt engineering provides a quick fix that might be enough for some simple use-cases, two main approaches have been developed in order to address these shortcomings - namely, <b>fine-tuning</b> and <b>retrieval augmented generation (RAG)</b>. Both of them imply adjusting and informing the LLM with specific data, so that its outputs can be steered toward our preferences. In this notebook, we will deep-dive into these two techniques, comparing them and highlighting their features.

</div>

### 1.2. Fine-tuning vs RAG - when to use which

<div style="width: 80%; text-align: justify">

<b>Fine-tuning</b> implies updating the model's parameters using task-specific data - think of it as a way to <b>communicate intent to the LLM</b>, so that the model can tailor its output to fit your goals. By training a pre-trained LLM on a smaller, more targeted data set, we can embed this new knowledge into the model's architecture. Thus, by communicating with the fine-tuned model afterwards, it can access this new knowledge in order to provide better responses.

On the other hand, <b>RAG</b> is a specific architectural framework that provides an LLM with access to an external knowledge base. Instead of modifying the model's parameters themselves through fine-tuning, RAG only modifies the prompt with relevant external data retreived from this knowledge base. This allows the LLM to incorporate up-to-date information from a source that we can control without retraining the underlying model. RAG essentially leverages in-context learning by providing additional context to the LLM's input query.

Here are some main differences between the two techniques:

</div>

| Criteria | Fine-tuning | RAG |
| --- | --- | --- |
| <b>Compute Requirements (Training)</b> | High <i>(depending on model size)</i> | None <i>(if using foundation models)</i> |
| <b>Compute Requirements (Inference)</b> | Normal <i>(data embedded in model)</i> | Slightly higher <i>(querying knowledge base)</i> |
| <b>Token Usage</b> | Normal | Higher <i>(data embedded in prompts)</i> |
| <b>Update Frequency</b> | Updating information implies retraining | Real-time information updates possible |
| <b>Data Requirements</b> | High <i>(for meaningful improvements)</i> | Low <i>(the more documents, the better)</i> |

<div style="width: 80%; text-align: justify">
<br/>

With all of this said, when should each technique be used?

<b>Fine-tuning is best for</b>:
- <b>Learning specific patterns or styles</b> - by fine-tuning on multiple input-output pairs, where outputs follow a certain style, the resulting LLM can learn and adopt it;

- Substantial, high-quality, <b>relatively static</b> data - retraining is needed for every information update;

- <b>Faster inference speeds</b> - by embedding the data directly in the model, we are not introducing any extra components such as vector stores that might introduce latency

<b>RAG is best for</b>:
- <b>Limited or dynamic, very current data</b> - the knowledge base can be as small as we want, and we can update it at any time without retraining the underlying LLM;

- <b>Citing sources directly for transparency</b> - by embedding the data in the prompt, it is easier for the model to output it directly if we prompt it to cite its sources;

- <b>Limited compute resources</b> - while RAG adds complexity at runtime, it avoids the considerable upfront cost of fine-tuning.

The two approaches can also be combined in a RAFT (Retrieval-Augmented Fine-Tuning) hybrid system, where fine-tuned models have access to an external knowledge base.

</div>

### 1.3. RAG

#### 1.3.1. RAG fundamentals - chunking, embedding, indexing

<div style="width: 80%; text-align: justify">

The RAG architecture fundamentally consists of three key components:
- <b>An external knowledge source</b> that stores our data;

- <b>A prompt template</b>, which provides a structured way to generate prompts with different sections - queries, context, etc.;

- <b>An LLM</b>, which is used to generate the final response;

<img src="https://i.imgur.com/QAnbvTZ.png" width="550px">

The RAG workflow operates in two main stages:

##### a. Ingestion Stage:
The ingestion stage marks the preparation of the external knowledge for efficient retrieval. This involves cleaning and transforming raw data into a format that the LLM can effectively use.

During ingestion, the data goes through multiple steps:
- <b>Chunking</b>: dividing large datasets into smaller, meaningful pieces. This process is central to the success of RAG workflows as it can improve efficiency and accuracy. The optimal chunking strategy depends on several factors, such as the document structure or task specificity. However, some popular chunking strategies are:
    - <b>Fixed-Size Chunking</b>: Simplest and most common approach, which splits text into uniformly sized segments based on a pre-defined number of characters; while easy to implement, it risks breaking context in the middle of the sentences, as it lacks semantic awareness;
    - <b>Recursive Chunking</b>: The text is split using a hierarchical set of separators (e.g. \n\n, \n, period, comma). If the initial split does not yield the desired chunk size, the method recursively applies finer splits until the target size is met.
    - <b>Semantic Chunking</b>: An advanced technique that divides documents into meaningful chunks based on the context rather than on an arbitrary size. It uses embeddings to group text based on semantic similarity, ensuring that semantically related sentences are kept together. This improves accuracy and relevancy significantly.

</div>

In [ ]:
loader = WebBaseLoader("https://ai-2027.com/")
raw_docs = loader.load()

web_content = [doc.page_content for doc in raw_docs]
web_metadata = [doc.metadata for doc in raw_docs]

print(f"Metadata: {web_metadata[0]}\n\nContent sample: {web_content[0][300:400]}")

Metadata: {'source': 'https://ai-2027.com/', 'title': 'AI 2027', 'description': 'A research-backed AI scenario forecast.\n', 'language': 'en'}

Content sample: xceeding that of the Industrial Revolution.We wrote a scenario that represents our best guess about 


In [ ]:
# Fixed-Size Chunking (Character Text Splitter)
fixed_size_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size=300,
    chunk_overlap=25,
    length_function=len,
    is_separator_regex=False
)

# Recursive Chunking
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=300,
    chunk_overlap=25
)

# Semantic Chunking
semantic_splitter = SemanticChunker(OpenAIEmbeddings(), breakpoint_threshold_type="percentile")

fixed_size_chunks = fixed_size_splitter.split_text(web_content[0])
recursive_chunks = recursive_splitter.split_text(web_content[0])
semantic_chunks = semantic_splitter.split_text(web_content[0])


Created a chunk of size 342, which is longer than the specified 300
Created a chunk of size 383, which is longer than the specified 300
Created a chunk of size 433, which is longer than the specified 300
Created a chunk of size 303, which is longer than the specified 300
Created a chunk of size 335, which is longer than the specified 300


Created a chunk of size 348, which is longer than the specified 300
Created a chunk of size 381, which is longer than the specified 300
Created a chunk of size 325, which is longer than the specified 300
Created a chunk of size 373, which is longer than the specified 300
Created a chunk of size 345, which is longer than the specified 300
Created a chunk of size 377, which is longer than the specified 300
Created a chunk of size 355, which is longer than the specified 300
Created a chunk of size 363, which is longer than the specified 300
Created a chunk of size 312, which is longer than the specified 300
Created a chunk of size 308, which is longer than the specified 300
Created a chunk of size 317, which is longer than the specified 300
Created a chunk of size 425, which is longer than the specified 300
Created a chunk of size 356, which is longer than the specified 300
Created a chunk of size 306, which is longer than the specified 300
Created a chunk of size 351, which is longer tha

In [ ]:
print(f"{len(fixed_size_chunks)} Fixed-Size Chunk(s): avg. length: {sum(len(chunk) for chunk in fixed_size_chunks) / len(fixed_size_chunks)}")
print(fixed_size_chunks[:5])
print(f"{len(recursive_chunks)} Recursive Chunk(s): avg. length: {sum(len(chunk) for chunk in recursive_chunks) / len(recursive_chunks)}")
print(recursive_chunks[:5])
print(f"{len(semantic_chunks)} Semantic Chunk(s): avg. length: {sum(len(chunk) for chunk in semantic_chunks) / len(semantic_chunks)}")
print(semantic_chunks[:5])

384 Fixed-Size Chunk(s): avg. length: 235.66927083333334
['AI 2027AI 2027SummaryResearchCompute ForecastTimelines ForecastTakeoff ForecastAI Goals ForecastSecurity ForecastAboutApril 3rd 2025 PDF ListenDaniel\xa0Kokotajlo, Scott\xa0Alexander, Thomas\xa0Larsen, Eli\xa0Lifland, Romeo\xa0DeanWe predict that the impact of superhuman AI over the next decade will be enormous, exceeding that of the Industrial Revolution', 'We wrote a scenario that represents our best guess about what that might look like.1 It’s informed by trend extrapolations, wargames, expert feedback, experience at OpenAI, and previous forecasting successes', '2What is this?How did we write it?Why is it valuable?Who are we?The CEOs of OpenAI, Google DeepMind, and Anthropic have all predicted that AGI will arrive within the next 5 years', 'Sam Altman has said OpenAI is setting its sights on “superintelligence in the true sense of the word” and the “glorious future.”3What might that look like? We wrote AI 2027 to answer that

<div style="width: 80%; text-align: justify">

- <b>Embedding</b>: the process of transforming text or image data from its raw format into numerical vector representations, known as embeddings. These high-dimensional vectors capture the semantic meaning of the data, allowing for contextual search and comparison. Embedding is done by an <b>embedding model</b>, a model that takes in text and projects it into a vector space. <b>It is essential to use the same embedding model for both embedding the data and querying the system</b>. Some important factors to keep in mind regarding this embedding model are:
    - <b>Training Data</b>: What data the embedding model was trained on. General-purpose data offers broad aplicability, while domain-specific data (e.g. legal) leads to better performance in niche applications.
    - <b>Embedding Size (Dimensionality)</b>: Higher dimensions capture more nuanced semantic information, but require more computational resources.
    - <b>Cost</b>: API usage fees for proprietary models or compute usage for self-hosting open-source models.
    - <b>MTEB Score</b>: The Massive Text Embedding Benchmark (MTEB) score measures a model's performance across various tasks, with higher scores indicating better overall quality. Up-to-date scores can be checked on [the following HuggingFace leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

</div>

In [ ]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings()

# Example texts with varying similarity to our query
query = "The benefits of exercise for mental health"
texts = [
    "Regular physical activity has been shown to reduce anxiety and depression while improving mood and cognitive function.",
    "Exercise releases endorphins and other neurotransmitters that help regulate emotions and reduce stress levels.",
    "Transformers are a type of neural network architecture that are particularly effective for natural language processing tasks.",
    "Cluj-Napoca is a city in Romania, famous for its historical center and the Transylvanian Saxon churches.",
]

# Get embeddings for query and texts
query_embedding = embeddings.embed_query(query)
text_embeddings = embeddings.embed_documents(texts)

# Calculate cosine similarities
similarities = []
for text_embedding in text_embeddings:
    similarity = dot(query_embedding, text_embedding)/(linalg.norm(query_embedding) * linalg.norm(text_embedding))
    similarities.append(similarity)

# Print results
print("\nQuery:", query)
print("\nSimilarities between query and texts:")
for text, similarity in zip(texts, similarities):
    print(f"\nText: {text}")
    print(f"Similarity score: {similarity:.3f}")



Query: The benefits of exercise for mental health

Similarities between query and texts:

Text: Regular physical activity has been shown to reduce anxiety and depression while improving mood and cognitive function.
Similarity score: 0.899

Text: Exercise releases endorphins and other neurotransmitters that help regulate emotions and reduce stress levels.
Similarity score: 0.863

Text: Transformers are a type of neural network architecture that are particularly effective for natural language processing tasks.
Similarity score: 0.710

Text: Cluj-Napoca is a city in Romania, famous for its historical center and the Transylvanian Saxon churches.
Similarity score: 0.678


<div style="width: 80%; text-align: justify">

- <b>Indexing</b>: storing the embeddings after they are generated, in a manner that allows for efficient and quick retrieval with high accuracy. This is achieved by indexing the models in a structured vector index database. We will talk about vector stores in depth in the following section.

##### b. Inference Stage:
After the external data is embedded and stored, it is ready to be retrieved during inference, which is when the model generates a response to a user query.

Starting with a user query, the workflow goes through the following steps:
- <b>Retrieval</b>: The user query is first embedded (<b>with the same model used for embedding the external data</b>), after which a similarity search is performed. Similarity search involves calculating similarities between the user query embedding and embeddings from the vector store. The data points whose embeddings are most similar with the user query embedding are passed to the following step.

- <b>Augmentation</b>: Once the most relevant information is retrieved from the external source, it is integrated into the prompt template. This effectively augments the LLM's prompt by providing contextually relevant information.

- <b>Generation</b>: The augmented prompt, now enriched with external information, is sent to the LLM. This model combines its internal language understanding with the newly retrieved external data, and generates a final response.

</div>

#### 1.3.2. Vector Stores

A vector store is a **specialized database** designed to efficiently store and retrieve high-dimensional vector data, particularly the embedddings generated from text or other modalities (e.g. images). These databases are integral to RAG systems, enabling powerful and context-aware search capabilities.

Vector stores can be **open-source** or **proprietary**. Moreover, in terms of hosting and deployment, they can be:
- **In-Memory** (e.g. FAISS): suitable for quick experimentation, but its data is not persistent;

- **Self-Hosted / Cloud-Managed** (e.g. Chroma, Qdrant, Weaviate): more robust solutions, with persistence and scalability;

- **Cloud-Only** (e.g. Pinecone, AstraDB): provided as managed services by cloud providers.

RAG workflows use **vector stores** because, given a **query vector** (the embedding of a user-provided text, e.g. a question), the vector store retrieve data points whose embeddings are closest to that query vector. To compare vectors, similarity measures are used. Common metrics, illustrated below, include:
- **Euclidean Distance**: Measures the straight-line path between two vectors, with 0 meaning identical vectors and larger numbers indicating greater separation.

- **Cosine Similarity**: Measures how two vectors point in the same direction, so it concerns the angle between the two vectors rather than the direct distance. The value ranges from -1 (opposite directions) to 1 (same direction).

- **Dot / Inner Product**: How well two vectors align, with positive values indicating alignment and 0 for orthogonality.

<img src="https://i.imgur.com/C9aytXn.png" width="750px">

A key feature of vector stores is **efficient storage and retrieval**. This is achieved by storing embeddings in a structured vector index database. A vector index is a data structure that enables fast and accurate search and retrieval of vector embeddings from large datasets. In order to build and search vector indexes, multiple **indexing methods** can be leveraged. Some widely-used examples are:
- **Flat Indexing**: Store each embedding location as is, mimicking a "map" data type. A search is done by **comparing the query vector to each vector in the store**. As expected, this approach implies a low memory usage, but search speed increases linearly with the number of vectors.

  For a query vector $ {q} $ and a dataset of vectors ${D=\{v_1, v_2, ..., v_n\}}$, a flat index computes $\text{similarity}(q, v_i)$ for all ${i}$ from 1 to ${n}$ and then sorts them to find the top ${k}$. This guarantees perfect accuracy (finding the actual nearest neighbors), but is computationally expensive, especially for large ${n}$.

  This approach does not scale well as the vector store increases to larger sizes, which is where <b>Approximate Nearest Neighbor (ANN)</b> methods come into play.
<hr/><br/>

- <b>Locality Sensitive Hashing (LSH) Indexes</b>: LSH maps similar vectors into the same buckets using a family of hash functions, enabling fast <b>approximate nearest neighbor search</b>. By limiting comparisons to only items in matching buckets, LSH significantly reduces computational complexity while trading off some accuracy.

    A hash function family ${h}$ is <b>locality sensitive</b> if it maps nearby points to the same value with high probability and distant points with low probability—that is, for any two points ${p}$ and ${q}$, ${\Pr[h(p) = h(q)]}$ is high when ${p \approx q}$ and low when ${p \not\approx q}$.

    During indexing, vectors are hashed into buckets. At query time, only vectors in the same bucket(s) as the query vector are compared, dramatically reducing the number of distance calculations.

    The diagram below highlights the LSH process:

    1. <b>Shingling</b>: Text is decomposed into character-level sequences ("shingles").

    2. <b>MinHashing</b>: Each shingle set is converted into a short signature. The probability that two MinHash values match equals their <b>Jaccard similarity</b>:
    $$
    \Pr[\text{MinHash}(A) = \text{MinHash}(B)] = \frac{|A \cap B|}{|A \cup B|}
    $$
    
    3. <b>Banding</b>: Signatures are split into bands of ${r}$ rows. Each band is hashed, and if two documents share any band, they become a <b>candidate pair</b> for further comparison.

    <img src="https://i.imgur.com/qFPBHxh.png" width="450px">
    <br/><br/>
    <hr/><br/>

- <b>Inverted File Indexes (IVF)</b>: IVF accelerates similarity search by partitioning the vector space into multiple clusters and restricting the search to only the most relevant ones. This reduces the number of distance computations and improves efficiency, especially in high-dimensional spaces.

    IVF begins with a <b>coarse quantization</b> step: the dataset is clustered into ${k}$ centroids using a method like k-means. Each data vector is assigned to the nearest centroid and stored in an <b>inverted list</b> (or "cell") corresponding to that centroid. During indexing, only the centroid ID and the residual vector (difference from the centroid) are stored.

    At query time:

    1. The query vector is compared to all centroids to find the top ${n_{probe}}$ closest centroids.
    2. Only the inverted lists corresponding to these ${n_{probe}}$ centroids are searched.
    3. Distance is computed between the query and vectors in these lists—optionally using residuals to improve precision.

    This process significantly reduces the number of comparisons from the full dataset size to a much smaller subset. IVF achieves a trade-off between <b>recall</b> and <b>efficiency</b>: larger ${n_{probe}}$ values increase accuracy but also the computational cost.

    <img src="https://i.imgur.com/O8Xa9gj.png" width="450px">
    <br/><br/>
    <hr/><br/>

- <b>Hierarchical Navigable Small World (HNSW) Graphs</b>: a graph-based indexing technique for fast and accurate <b>approximate nearest neighbor (ANN)</b> search. It organizes data into a layered structure of proximity graphs, where each layer is a sparse graph built using the <b>small-world</b> principle—few long-range links plus many short-range ones.

    Each data point is assigned a maximum layer level ${l}$ (sampled randomly with exponential decay), and added to all layers from level 0 up to ${l}$. The top layer contains very few points, while layer 0 contains all points.

     HNSW leverages three key principles for efficient search. First, its **multi-layered structure** enables logarithmic search paths with approximately $O(\log N)$ hops to reach the query region, where $N$ is the number of indexed vectors. Second, each layer forms a **navigable small-world graph** with logarithmic diameter and polylogarithmic node degree, resulting in $O(\log^k N)$ expected nodes visited during search (where $k \approx 1.2-1.4$ in practice).

     Finally, **probabilistic level assignment** uses a geometric distribution $\Pr[\text{level} \geq l] = e^{-\lambda l}$ to determine each node's maximum layer, ensuring sparse upper layers and dense base coverage. This combination creates an efficient hierarchical structure that balances search speed and accuracy.

    <b>The search procedure can be seen in the following diagram</b>:

    1. Begin from an <b>entry point</b> in the topmost layer.
    2. At each layer, navigate greedily: move to the neighbor closest to the query until no improvement is possible.
    3. Descend one layer and repeat the search from the current best node.
    4. At layer 0, perform a more extensive search within a fixed-size candidate list to refine the result.


    <img src="https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2Fe63ca5c638bc3cd61cc1cd2ab33b101d82170426-1920x1080.png&w=3840&q=75" width="450px"/>

#### 1.3.3. RAG - Putting it all together (LangChain + LanceDB)

<div style="width: 80%; text-align: justify">

In order to see how a RAG workflow is built from the ground up, we'll dive into an example that combines LangChain agents (which we've covered in the previous notebook) with the LanceDB vector store. Our project will demonstrate a complete RAG pipeline by collecting data from a website (such as AI 2027), creating a vector store from the scraped content, and building an intelligent Q&A system that can answer questions about the collected information.

<b>LanceDB</b> is an open-source vector database specifically designed for AI workloads. It is built on Lance, a columnar data format written in Rust, optimized for fast access and high performance. LanceDB offers several key advantages for RAG applications:

- **Embedded and serverless**: Available as an open-source embedded database that runs in-process, making self-hosting straightforward with zero configuration;

- **Flexible storage**: Supports multiple storage backends including local disk, cloud solutions (S3 or EFS on AWS, Blob Storage or File Storage on Azure), and remote databases;

- **Multi-modal support**: Can store and search not just text embeddings, but also images, videos, and other data types;

- **ACID transactions**: Provides data consistency and reliability for production workloads;

- **SQL interface**: Offers familiar SQL querying capabilities alongside vector search;

- **Automatic versioning**: Built-in data versioning and time-travel capabilities for reproducible experiments;

</div>

In [ ]:
loader = WebBaseLoader("https://ai-2027.com/")
raw_docs = loader.load()

web_content = [doc.page_content for doc in raw_docs]
web_metadata = [doc.metadata for doc in raw_docs]

print(f"Extracted {len(raw_docs)} documents from the website")
print(f"Metadata: {web_metadata[0]}\n\nContent sample: {web_content[0][300:400]}")

Extracted 1 documents from the website
Metadata: {'source': 'https://ai-2027.com/', 'title': 'AI 2027', 'description': 'A research-backed AI scenario forecast.\n', 'language': 'en'}

Content sample: xceeding that of the Industrial Revolution.We wrote a scenario that represents our best guess about 


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", "!", "?", ";", ":", ",", " "],
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(raw_docs)

n_docs = len(docs)
avg_doc_length = int(sum(len(doc.page_content) for doc in docs) / n_docs)

print(f"Split {len(raw_docs)} documents into {n_docs} chunks with avg. length {avg_doc_length}")

random_doc = random.choice(docs)
print(f"Chunk sample metadata: {random_doc.metadata}")
print(f"Chunk sample content: {random_doc.page_content[:300]}")

Split 1 documents into 116 chunks with avg. length 913
Chunk sample metadata: {'source': 'https://ai-2027.com/', 'title': 'AI 2027', 'description': 'A research-backed AI scenario forecast.\n', 'language': 'en'}
Chunk sample content: . Nobody has a crystal ball, but this type of content can help notice important questions and illustrate the potential impact of emerging risks.” —Yoshua Bengio7We have set ourselves an impossible task. Trying to predict how superhuman AI in 2027 would go is like trying to predict how World War 3 in


In [ ]:
!rm -rf resources/vector_store

/Users/admin.lateral/ai_academy/.venv/lib/python3.12/site-packages/lancedb/__init__.py:220: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn instead.
  warnings.warn(


In [ ]:
lancedb_connection = lancedb.connect("resources/vector_store")
embeddings = OpenAIEmbeddings()

vector_store = LanceDB.from_documents(
    documents=docs,
    embedding=embeddings,
    connection=lancedb_connection
)

In [ ]:
test_query = "What will be China's role in the AI industry between 2025 and 2027?"

results = vector_store.similarity_search_with_relevance_scores(test_query)
print(f"Found {len(results)} results")

for doc, score in results:
    print(f"Content: {doc.page_content}")
    print(f"Relevance score: {score:.3f}")
    print("\n")

Found 4 results
Content: . In early 2025, the worst-case scenario was leaked algorithmic secrets; now, if China steals Agent-1’s weights, they could increase their research speed by nearly 50%.31 OpenBrain’s security level is typical of a fast-growing ~3,000 person tech company, secure only against low-priority attacks from capable cyber groups (RAND’s SL2).32 They are working hard to protect their weights and secrets from insider threats and top cybercrime syndicates (SL3),33 but defense against nation states (SL4&5) is barely on the horizon.Mid 2026: China Wakes UpIn China, the CCP is starting to feel the AGI.Chip export controls and lack of government support have left China under-resourced compared to the West. By smuggling banned Taiwanese chips, buying older chips, and producing domestic chips about three years behind the US-Taiwanese frontier, China has managed to maintain about 12% of the world’s AI-relevant compute—but the older technology is harder to work with, and supply is

In [ ]:
model = init_chat_model(
    model="gpt-4o-mini"
)

context_text = '\n'.join((doc.page_content for doc, _ in results))

rag_prompt = f"""
Based on the following context, answer the question.

Context:
{context_text}

Question:
{test_query}
"""

response = model.invoke(rag_prompt)


In [ ]:
print(response.content)

Between 2025 and 2027, China's role in the AI industry is characterized by a significant push to overcome its technological deficits and assert its position in the race for artificial general intelligence (AGI). Despite facing challenges such as chip export controls, resource limitations, and reliance on older technology, the Chinese Communist Party (CCP) recognizes the urgency of advancing in AI research.

1. **Aggressive AI Development**: The CCP commits to a major AI push, nationalizing AI research and creating an information-sharing mechanism among AI companies. This is aimed at consolidating resources, datasets, and algorithmic insights to accelerate advancements in the field.

2. **Centralization of Compute Resources**: By restructuring its AI landscape, China centralizes a growing share of AI-relevant compute, significantly increasing its capacity. By 2027, it is suggested that up to 70% of this compute could be controlled within a designated zone (CDZ).

3. **Collaboration and 

#### 1.3.4. Evaluating RAG pipelines

<div style="width: 80%; text-align: justify">

Several frameworks and tools facilitate the evaluation of RAG pipelines. In this notebook, we'll take a look at <b>Ragas</b>, an open-source library that uses LLM-powered metrics. Thus, human-labeled datasets are not needed:

</div>

In [ ]:
sample_docs = [
    "Lateral is a company with offices in many cities, including Cluj-Napoca, Romania.",
    "Lateral was founded in 2008, in Targu-Mures, Romania.",
    "The AI Academy 2025 program started on the 7th of July, 2025, and is for a duration of 8 weeks.",
    "In late May 2025, heavy rains in Romania caused floods and landslides, affecting many areas. A notable incident was the collapse of Salina Praid, a salt mine in Harghita.",
    "Due to a high general government deficit (9.3 percent of GDP in 2024), Romania's economic forecast is negative, with heavy cuts and tax increases expected.",
]

sample_queries = [
    "Does Lateral have any offices in Romania? If so, which cities?",
    "What year was Lateral founded and in which Romanian city?",
    "What is the duration of the AI Academy 2025 program and when did it begin?",
    "What natural disasters occurred in Romania in May 2025, and what specific infrastructure was affected?",
    "What was Romania's government deficit as a percentage of GDP in 2024, and what economic measures are expected as a result?"
]

expected_responses = [
    "Yes, Lateral has offices in Cluj-Napoca, Romania, and the document mentions they have offices in many cities.",
    "Lateral was founded in 2008 in Targu-Mures, Romania.",
    "The AI Academy 2025 program is for a duration of 8 weeks and started on the 7th of July, 2025.",
    "In late May 2025, heavy rains in Romania caused floods and landslides. A notable incident was the collapse of Salina Praid, a salt mine in Harghita.",
    "Romania had a high general government deficit of 9.3 percent of GDP in 2024, which has led to a negative economic forecast with heavy cuts and tax increases expected."
]

In [ ]:
class RAG:
    def __init__(self, model="gpt-4o"):
        self.llm = init_chat_model(model)
        self.embeddings = OpenAIEmbeddings()
        self.doc_embeddings = None
        self.docs = None

    def load_documents(self, docs):
        self.docs = docs
        self.doc_embeddings = self.embeddings.embed_documents(self.docs)

    def find_relevant_doc(self, query):
        if not self.docs or not self.doc_embeddings:
            raise ValueError("Documents or embeddings not loaded. Please call load_documents first.")

        query_embedding = self.embeddings.embed_query(query)
        similarities = [
            np.dot(doc_emb, query_embedding)
            / (np.linalg.norm(doc_emb) * np.linalg.norm(query_embedding))
            for doc_emb in self.doc_embeddings
        ]
        most_relevant_doc_ind = np.argmax(similarities)
        return [self.docs[most_relevant_doc_ind]]

    def generate_answer(self, context, query):
        prompt = f"""
        You are a helpful assistant that can answer questions based on provided context.
        Context:
        {context}

        Question:
        {query}
        """

        response = self.llm.invoke(prompt)
        return response.content

rag = RAG()
rag.load_documents(sample_docs)

query = "What is the economic forecast for Romania?"

context = rag.find_relevant_doc(query)
answer = rag.generate_answer(context, query)

print(answer)


The economic forecast for Romania is negative due to a high general government deficit, which is projected to be 9.3 percent of GDP in 2024. Consequently, heavy cuts and tax increases are expected.


In [ ]:
dataset = []

for query, expected_response in zip(sample_queries, expected_responses):
    relevant_doc = rag.find_relevant_doc(query)
    response = rag.generate_answer(relevant_doc, query)
    dataset.append(
        {
            "user_input": query,
            "retrieved_contexts": relevant_doc,
            "response": response,
            "reference": expected_response
        }
    )

In [ ]:
eval_dataset = EvaluationDataset.from_list(dataset)

In [ ]:
evaluator_llm = LangchainLLMWrapper(rag.llm)

In [ ]:
result = evaluate(
    dataset=eval_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness()
    ],
    llm=evaluator_llm
)

Evaluating: 100%|██████████| 15/15 [00:09<00:00,  1.54it/s]


In [ ]:
result

{'context_recall': 1.0000, 'faithfulness': 1.0000, 'factual_correctness(mode=f1)': 0.8080}

# 2. Assignment

<div style="width: 80%; text-align: justify">

<b>Objective</b>

Augment the agent pipeline from your previous assignment, extending it with a vector store and RAG functionality.

1. Pick 5-10 plant classes and search some care guides for them on the web.
2. Use Langchain loaders & splitters to turn these webpages into Documents.
3. Build up a vector store (e.g. LanceDB) from your Documents, and set up tools for querying it.
4. Augment your existing pipeline - query the vector store using your plant class label, and augment the prompt with additional context before sending it to the agent.
5. (Optional) Modify your structured output - try to add some fields that the LLM couldn't fill accurately without external information (e.g. is it friendly for pets?)
</div>

<b>Deliverable</b>
- Your notebook containing the final pipeline, as well as code used for building up & querying the vector store.
- Some example outputs (e.g. pictures of aloe vera, and their generated care cards).

# 3. Further Reading

- <b>Finetuning & RAG</b>

    - <b>[Awesome RAG - GitHub](https://github.com/Danielskry/Awesome-RAG)</b> - A curated list of awesome RAG resources.

    - <b>[LocalLLaMA subreddit](https://www.reddit.com/r/LocalLLaMA/)</b> - Some general LLM discussions; also includes topics like fine-tuning and RAG.

    - <b>[ZenML Database](https://www.zenml.io/llmops-database)</b> - A database of useful LLMOps workflows; includes some nice RAG workflows.